# Hotel Dynamic Pricing using Reinforcement Learning (Q-Learning)

This notebook demonstrates how to initialize the pricing environment, fit the historical demand models, train a Q-learning agent, and evaluate its performance against baseline strategies.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from src.config import FEATURES_DATA_PATH, PRICE_ACTIONS
from src.rl.demand_simulator import DemandSimulator
from src.rl.environment import HotelPricingEnvironment
from src.rl.q_learning import QLearningAgent
from src.rl.evaluator import run_evaluation_flow

## Step 1: Initialize and Fit Demand Simulator
We load the processed dataset `data/processed/hotel_bookings_features.csv` and train the demand models (expected price regressor and booking confirmation classifier).

In [ ]:
# Instantiate and fit Demand Simulator
demand_sim = DemandSimulator(data_path=FEATURES_DATA_PATH)
demand_sim.fit()

## Step 2: Create Environment and Agent
We initialize the pricing environment and the tabular Q-learning agent.

In [ ]:
# Create environment
env = HotelPricingEnvironment(
    data_path=str(FEATURES_DATA_PATH),
    demand_simulator=demand_sim
)

# Create Q-learning agent
agent = QLearningAgent(
    price_actions=env.price_actions,
    max_inventory=env.max_inventory,
    max_days=env.max_days
)

## Step 3: Train the Q-Learning Agent
We run the training loop for the agent over 1000 episodes, saving performance metrics.

In [ ]:
# Temporarily adjust logging levels to avoid notebook clutter
import logging
logging.getLogger("q_learning").setLevel(logging.WARNING)
logging.getLogger("environment").setLevel(logging.WARNING)

df_metrics = agent.train(env, episodes=1000)
df_metrics.head()

## Step 4: Evaluate Q-Learning against Baselines
We compare Q-learning against Fixed Pricing, Discount Pricing, and Random Pricing policies over 100 evaluation episodes.

In [ ]:
# Restore logging and run evaluation comparison
logging.getLogger("evaluator").setLevel(logging.INFO)
df_summary = run_evaluation_flow(q_agent=agent, env=env, episodes=100)
df_summary

## Step 5: Visualize the Results
We display the generated performance plots.

In [ ]:
from IPython.display import Image, display

figures_to_show = [
    "learning_curve.png",
    "revenue_comparison.png",
    "occupancy_comparison.png",
    "q_table_heatmap.png",
    "price_distribution.png"
]

figures_dir = Path("../outputs/figures")
for fig_name in figures_to_show:
    fig_path = figures_dir / fig_name
    if fig_path.exists():
        print(f"\n--- Displaying {fig_name} ---")
        display(Image(filename=str(fig_path)))
    else:
        # Check current directory fallback
        fallback_path = Path("outputs/figures") / fig_name
        if fallback_path.exists():
            print(f"\n--- Displaying {fig_name} ---")
            display(Image(filename=str(fallback_path)))